In [ ]:
import os, sys
# Add project root to sys.path so we can import models and pipelines
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)


# PRISM: Embedding Extraction (Sprint 4)
This notebook extracts 512-dim embeddings from **all 131K segments** using the trained CNN checkpoint.

**Prerequisites:**
1. `prism-colab/` folder on Google Drive (with the updated `models/embeddings/extract_embeddings.py`)
2. `datasets-features.zip` on Google Drive (same file from CNN training — no re-upload needed)
3. `cough_detector_best.pt` in `prism-colab/checkpoints/` on Google Drive

**Output:**
- `embeddings_matrix.npy` — shape (N, 512), ~256 MB
- `embeddings_metadata.csv` — row-aligned segment metadata

In [ ]:
# === Cell 1: Setup Environment and Mount Drive ===
!pip install -q loguru rich scikit-learn pyyaml tqdm

import os

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Symlink the PRISM code into the Colab working directory
if not os.path.exists('/content/prism'):
    os.symlink('/content/drive/MyDrive/prism-colab', '/content/prism')
if not os.path.exists('/content/models'):
    os.symlink('/content/drive/MyDrive/prism-colab/models', '/content/models')

print("\n\u2705 Code setup complete!")

In [ ]:
# === Cell 2: Unzip Features (Run ONLY ONCE per session) ===
import os

if not os.path.exists('/content/features/manifest.csv'):
    print("\u23f3 Unzipping features to local SSD... This will take a few minutes.")
    !unzip -q /content/drive/MyDrive/datasets-features.zip -d /content/features
    print("\u2705 Features unzipped!")
else:
    print("\u2705 Features already unzipped.")

In [ ]:
# === Cell 3: Verify GPU ===
import torch

if torch.cuda.is_available():
    print(f"\U0001f680 GPU Active: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("\u274c ERROR: No GPU found. Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

In [ ]:
# === Cell 4: Extract Embeddings ===
import os

os.chdir('/content/prism')

!python -m models.embeddings.extract_embeddings \
    --checkpoint /content/drive/MyDrive/prism-colab/checkpoints/cough_detector_best.pt \
    --manifest /content/features/datasets/features/manifest.csv \
    --features-dir /content/features/datasets/features \
    --output-dir /content/embeddings_output \
    --batch-size 256 \
    --num-workers 2

In [ ]:
# === Cell 5: Verify Output ===
import numpy as np
import pandas as pd

matrix = np.load('/content/embeddings_output/embeddings_matrix.npy')
metadata = pd.read_csv('/content/embeddings_output/embeddings_metadata.csv')

print(f"\u2705 Embeddings matrix shape: {matrix.shape}")
print(f"\u2705 Metadata rows: {len(metadata)}")
print(f"\u2705 Matrix size: {matrix.nbytes / 1e6:.1f} MB")
print(f"\u2705 Rows match: {matrix.shape[0] == len(metadata)}")
print("\n--- Sample metadata ---")
print(metadata.head(10))

# Quick sanity: check L2 norms are ~1.0 (should be normalised)
norms = np.linalg.norm(matrix[:100], axis=1)
print("\n--- L2 norm stats (first 100) ---")
print(f"Mean: {norms.mean():.4f} | Std: {norms.std():.6f} | Min: {norms.min():.4f} | Max: {norms.max():.4f}")

In [ ]:
# === Cell 6: Copy Results to Google Drive ===
import os
import shutil

drive_output = '/content/drive/MyDrive/prism-colab/embeddings'
os.makedirs(drive_output, exist_ok=True)

shutil.copy2('/content/embeddings_output/embeddings_matrix.npy', drive_output)
shutil.copy2('/content/embeddings_output/embeddings_metadata.csv', drive_output)

print("\u2705 Embeddings saved to Google Drive!")
print(f"   {drive_output}/embeddings_matrix.npy")
print(f"   {drive_output}/embeddings_metadata.csv")
print("\nYou can now download these to your local machine:")
print("   -> Place them in: PRISM/models/embeddings/")